# PostgreSQL table row counts

This notebook checks row counts in tables created from the adapted scripts.

It includes:
- counts for all tables in the target schema
- focused counts for key test/data tables

In [27]:
import os
from pathlib import Path

import psycopg
from psycopg.rows import dict_row


def load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env(Path.cwd().parent / ".env")

DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE"),
    "user": os.getenv("PGUSER"),
    "password": os.getenv("PGPASSWORD"),
}
SCHEMA = "s_grnplm_as_t_didsd_nnn_db_tmd"

conn = psycopg.connect(**DB_CONFIG, row_factory=dict_row)
print("Connected")

Connected


In [28]:
# List tables in schema
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = %s
        ORDER BY table_name
        """,
        (SCHEMA,),
    )
    tables = [r["table_name"] for r in cur.fetchall()]

print(f"Found {len(tables)} tables")
tables

Found 25 tables


['d_agr_cred',
 'd_agr_cred_1497470240',
 'd_agr_cred_coa',
 'd_agr_cred_coa_1497470240',
 'd_agr_cred_coa_period_calc_dt',
 'd_agr_cred_coa_period_prep_bal',
 'd_agr_cred_coa_period_prep_bal_531401232',
 'd_agr_cred_coa_period_prep_main_debts',
 'd_agr_cred_coa_period_prep_main_debts_531401232',
 'd_agr_cred_optn',
 'd_agr_cred_optn_1497470240',
 'd_agr_cred_prvsn_period',
 'd_settings',
 'd_tech_tbl_coa_bal_h',
 'd_tech_tbl_coa_bal_h_1497470240',
 'debug_log',
 'etl_bkmart_log',
 'etl_task_param',
 't_coa',
 't_coa_129171027',
 't_je_line',
 't_je_line_129171027',
 't_spt_je_header_fltr',
 't_src_system_type',
 't_str_pilot_tune']

In [29]:
# Row counts for all tables in schema
all_counts = []
with conn.cursor() as cur:
    for table_name in tables:
        full_name = f'"{SCHEMA}"."{table_name}"'
        cur.execute(f"SELECT count(*) AS cnt FROM {full_name}")
        all_counts.append({"table": table_name, "row_count": cur.fetchone()["cnt"]})

# Sort descending by row count for quick inspection
all_counts = sorted(all_counts, key=lambda x: x["row_count"], reverse=True)
all_counts

[{'table': 'd_agr_cred_optn', 'row_count': 20000000},
 {'table': 'd_agr_cred_optn_1497470240', 'row_count': 20000000},
 {'table': 'd_tech_tbl_coa_bal_h', 'row_count': 2000000},
 {'table': 'd_tech_tbl_coa_bal_h_1497470240', 'row_count': 2000000},
 {'table': 't_je_line', 'row_count': 1000000},
 {'table': 't_je_line_129171027', 'row_count': 1000000},
 {'table': 'd_agr_cred', 'row_count': 200000},
 {'table': 'd_agr_cred_1497470240', 'row_count': 200000},
 {'table': 'd_agr_cred_coa', 'row_count': 200000},
 {'table': 'd_agr_cred_coa_1497470240', 'row_count': 200000},
 {'table': 'd_agr_cred_coa_period_calc_dt', 'row_count': 100000},
 {'table': 'd_agr_cred_coa_period_prep_bal', 'row_count': 100000},
 {'table': 'd_agr_cred_coa_period_prep_bal_531401232', 'row_count': 100000},
 {'table': 'd_agr_cred_coa_period_prep_main_debts', 'row_count': 100000},
 {'table': 'd_agr_cred_coa_period_prep_main_debts_531401232',
  'row_count': 100000},
 {'table': 'd_agr_cred_prvsn_period', 'row_count': 100000},
 {

In [30]:
# Focused checks for key tables
key_tables = [
    "t_coa",
    "t_je_line",
    "t_spt_je_header_fltr",
    "etl_bkmart_log",
    "d_agr_cred",
    "d_agr_cred_coa",
    "d_agr_cred_optn",
    "d_agr_cred_coa_period_prep_bal",
    "d_agr_cred_coa_period_prep_main_debts",
    "d_agr_cred_coa_period_calc_dt",
]

key_counts = []
with conn.cursor() as cur:
    for table_name in key_tables:
        cur.execute(
            """
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE table_schema = %s AND table_name = %s
            ) AS exists_flag
            """,
            (SCHEMA, table_name),
        )
        exists_flag = cur.fetchone()["exists_flag"]

        if exists_flag:
            full_name = f'"{SCHEMA}"."{table_name}"'
            cur.execute(f"SELECT count(*) AS cnt FROM {full_name}")
            cnt = cur.fetchone()["cnt"]
        else:
            cnt = None

        key_counts.append({"table": table_name, "exists": exists_flag, "row_count": cnt})

key_counts

[{'table': 't_coa', 'exists': True, 'row_count': 3000},
 {'table': 't_je_line', 'exists': True, 'row_count': 1000000},
 {'table': 't_spt_je_header_fltr', 'exists': True, 'row_count': 100000},
 {'table': 'etl_bkmart_log', 'exists': True, 'row_count': 103},
 {'table': 'd_agr_cred', 'exists': True, 'row_count': 200000},
 {'table': 'd_agr_cred_coa', 'exists': True, 'row_count': 200000},
 {'table': 'd_agr_cred_optn', 'exists': True, 'row_count': 20000000},
 {'table': 'd_agr_cred_coa_period_prep_bal',
  'exists': True,
  'row_count': 100000},
 {'table': 'd_agr_cred_coa_period_prep_main_debts',
  'exists': True,
  'row_count': 100000},
 {'table': 'd_agr_cred_coa_period_calc_dt',
  'exists': True,
  'row_count': 100000}]

In [31]:
conn.close()
print("Connection closed")

Connection closed
